# 후쿠오카 숙박업소 리뷰 감성분석 — Google Colab

## 실행 순서
1. **메뉴 → 런타임 → 런타임 유형 변경 → GPU(T4) 선택** (먼저 하세요!)
2. 셀을 위에서 아래로 순서대로 실행 (셀 왼쪽 ▶ 버튼 클릭)
3. **[2단계]** 에서 `full_cleaned.csv` 파일을 업로드해야 합니다

---
### 이 노트북이 하는 일
- **[1~2단계]** 환경 설정 및 파일 업로드
- **[3단계]** 5종 모델 비교 검증 (긍정/부정/중립 정확도 측정)
- **[4단계]** 최종 모델로 81,613건 전체 감성분석 실행
- **[5단계]** 결과 다운로드

## [1단계] 패키지 설치
처음 한 번만 실행하면 됩니다. 1~2분 소요됩니다.

In [ ]:
!pip install -q transformers torch pandas tqdm python-dateutil
print('✅ 패키지 설치 완료')

## [2단계] GPU 확인 + 데이터 파일 업로드
이 셀을 실행하면 파일 업로드 버튼이 나타납니다.
`full_cleaned.csv` (81,613건 전처리 완료 파일)를 업로드해주세요.

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'디바이스: {device}')
if device == 'cpu':
    print('⚠️  GPU가 감지되지 않았습니다. 메뉴 → 런타임 → 런타임 유형 변경 → GPU(T4) 로 바꿔주세요.')
else:
    print(f'✅ GPU 사용 가능: {torch.cuda.get_device_name(0)}')

from google.colab import files
print('\nfull_cleaned.csv 파일을 업로드해주세요.')
uploaded = files.upload()
print('\n업로드된 파일:', list(uploaded.keys()))

In [ ]:
import pandas as pd
df_full = pd.read_csv('full_cleaned.csv')
print(f'✅ 데이터 로드 완료: {len(df_full):,}건')
print(f'컬럼: {list(df_full.columns)}')
df_full.head(3)

## [3단계] 5종 모델 비교 검증
각 모델이 호텔 리뷰 문장을 얼마나 정확히 분류하는지 측정합니다.
모델 5개를 순서대로 다운로드하면서 테스트하니 **10~20분** 소요됩니다.

마지막에 **비교 요약표**가 출력됩니다.

In [ ]:
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

CANDIDATES = [
    {
        'name': 'alsgyu/sentiment-analysis-fine-tuned-model',
        'desc': 'KcBERT | AI허브 감정 데이터 | 3분류 (기검증)',
        'label_map': {'LABEL_0': '부정', 'LABEL_1': '중립', 'LABEL_2': '긍정'},
        'pos_key': 'LABEL_2', 'neg_key': 'LABEL_0',
    },
    {
        'name': 'Copycats/koelectra-base-v3-generalized-sentiment-analysis',
        'desc': 'KoELECTRA-v3 | 쇼핑/상품 리뷰 | 2분류',
        'label_map': None, 'pos_key': None, 'neg_key': None,
    },
    {
        'name': 'WhitePeak/bert-base-cased-Korean-sentiment',
        'desc': 'Korean BERT | F1 92.3% | 2분류',
        'label_map': None, 'pos_key': None, 'neg_key': None,
    },
    {
        'name': 'snunlp/KR-FinBert-SC',
        'desc': 'FinBERT | 금융 뉴스 도메인 | 3분류 (비교군 — 이전 실패 모델)',
        'label_map': {'positive': '긍정', 'negative': '부정', 'neutral': '중립'},
        'pos_key': 'positive', 'neg_key': 'negative',
    },
    {
        'name': 'monologg/koelectra-base-finetuned-sentiment',
        'desc': 'KoELECTRA | NSMC 영화리뷰 | 2분류',
        'label_map': None, 'pos_key': None, 'neg_key': None,
    },
]

POS = [
    '위치 객실 전망 주변편의시설 모든게 다 좋았습니다 조식도 맛있었어요',
    '직원분들도 너무 친절하고 아늑한 느낌의 호텔이었어요 다음에 또 올게요',
    '가족여행으로 갔는데 부모님이 아주 좋아하셨습니다 강력 추천합니다',
    '깨끗하고 조용해서 푹 쉬다 갑니다 다음에도 꼭 다시 오고 싶어요',
    '가격도 합리적이고 위치도 최고예요 정말 만족스러운 숙박이었습니다',
]
NEG = [
    '방이 너무 좁고 곰팡이 냄새가 나서 최악이었습니다 환불해주세요',
    '직원이 너무 불친절하고 시설도 낙후되어 실망했습니다',
    '예약한 방과 다른 방을 줘서 너무 황당했고 사과도 받지 못했습니다',
    '벌레가 나와서 한숨도 못 잤어요 절대 비추천입니다',
    '청소 상태가 엉망이고 냄새가 심해서 다시는 안 올 것 같아요',
]
NEU = [
    '체크인은 3시이고 체크아웃은 11시입니다',
    '호텔 1층에 편의점이 있고 근처에 편의시설이 있습니다',
    '역에서 도보 10분 거리에 위치해 있습니다',
]


def auto_detect(model_name, id2label):
    tok = AutoTokenizer.from_pretrained(model_name)
    mod = AutoModelForSequenceClassification.from_pretrained(model_name)
    mod.eval()
    pos_scores = {v: 0.0 for v in id2label.values()}
    neg_scores = {v: 0.0 for v in id2label.values()}
    for t in POS[:3]:
        inp = tok(t, return_tensors='pt', truncation=True, max_length=128)
        with torch.no_grad():
            p = torch.softmax(mod(**inp).logits, dim=-1).numpy()[0]
        for i, v in enumerate(p): pos_scores[id2label[i]] += float(v)
    for t in NEG[:3]:
        inp = tok(t, return_tensors='pt', truncation=True, max_length=128)
        with torch.no_grad():
            p = torch.softmax(mod(**inp).logits, dim=-1).numpy()[0]
        for i, v in enumerate(p): neg_scores[id2label[i]] += float(v)
    pos_key = max(pos_scores, key=pos_scores.get)
    neg_key = max(neg_scores, key=neg_scores.get)
    label_map = {k: ('긍정' if k==pos_key else '부정' if k==neg_key else '중립')
                 for k in id2label.values()}
    return label_map, pos_key, neg_key


def test_one(cfg):
    name = cfg['name']
    print(f'\n{"-"*60}')
    print(f'모델: {name.split("/")[-1]}')
    print(f'설명: {cfg["desc"]}')
    try:
        tok = AutoTokenizer.from_pretrained(name)
        mod = AutoModelForSequenceClassification.from_pretrained(name)
        mod.eval()
    except Exception as e:
        print(f'  ❌ 로드 실패: {e}')
        return {'name': name, 'acc': None, 'status': '로드 실패'}

    id2label = mod.config.id2label
    label_map = cfg['label_map']
    pos_key, neg_key = cfg['pos_key'], cfg['neg_key']
    if label_map is None:
        label_map, pos_key, neg_key = auto_detect(name, id2label)
        print(f'  [자동 탐지] pos={pos_key}, neg={neg_key}')
    print(f'  라벨 매핑: {label_map}')

    label_keys = [id2label[i] for i in range(len(id2label))]

    def predict(text):
        inp = tok(text, return_tensors='pt', truncation=True, max_length=128)
        with torch.no_grad():
            p = torch.softmax(mod(**inp).logits, dim=-1).numpy()[0]
        idx = int(np.argmax(p))
        return label_map.get(id2label[idx], id2label[idx]), float(p[idx])

    correct, total = 0, 0
    for group, samples, expected in [('긍정', POS, '긍정'), ('부정', NEG, '부정'), ('중립(참고)', NEU, None)]:
        print(f'  [{group}]')
        for s in samples:
            pred, score = predict(s)
            if expected:
                ok = pred == expected
                correct += int(ok)
                total += 1
                mark = '✅' if ok else '❌'
            else:
                mark = '  '
            print(f'    {mark} {pred:4s} {score:.3f} | {s[:40]}')

    acc = correct / total if total else 0
    print(f'  >>> 정확도: {correct}/{total} ({acc*100:.0f}%)')
    return {'name': name, 'acc': acc, 'label_map': label_map,
            'pos_key': pos_key, 'neg_key': neg_key, 'status': '성공'}


summary = []
for cfg in CANDIDATES:
    result = test_one(cfg)
    summary.append(result)

print('\n' + '='*60)
print('최종 비교 요약')
print('='*60)
print(f'{"#":<3} {"모델":<48} {"정확도":>6}  상태')
print('-'*60)
best = None
for i, r in enumerate(summary, 1):
    short = r['name'].split('/')[-1][:46]
    acc_str = f"{r['acc']*100:.0f}%" if r['acc'] is not None else 'N/A'
    print(f'{i:<3} {short:<48} {acc_str:>6}  {r["status"]}')
    if r['acc'] and (best is None or r['acc'] > best['acc']):
        best = r

if best:
    print(f'\n✅ 추천 모델: {best["name"]} (정확도 {best["acc"]*100:.0f}%)')
    print(f'   라벨 매핑: {best["label_map"]}')
    BEST_MODEL = best['name']
    BEST_LABEL_MAP = best['label_map']
    BEST_POS_KEY = best['pos_key']
    BEST_NEG_KEY = best['neg_key']
    print('\n위 값이 [4단계]에 자동으로 적용됩니다. 바로 [4단계] 셀을 실행해주세요.')

## [4단계] 최종 모델로 81,613건 전체 감성분석
[3단계]에서 자동으로 선택된 최적 모델로 전체 데이터를 분석합니다.
GPU(T4) 기준 약 20~40분 소요됩니다.

In [ ]:
from tqdm.notebook import tqdm
import time

print(f'사용 모델: {BEST_MODEL}')
print(f'라벨 매핑: {BEST_LABEL_MAP}')
print(f'분석 대상: {len(df_full):,}건')

BATCH_SIZE = 32

tok = AutoTokenizer.from_pretrained(BEST_MODEL)
mod = AutoModelForSequenceClassification.from_pretrained(BEST_MODEL)
mod.eval()
mod.to(device)
print(f'\n✅ 모델 로드 완료 ({device})')

id2label = mod.config.id2label
label_keys = [id2label[i] for i in range(len(id2label))]

texts = df_full['text'].tolist()
sentiments, scores, score_pos_list, score_neg_list = [], [], [], []

pos_idx = label_keys.index(BEST_POS_KEY) if BEST_POS_KEY in label_keys else None
neg_idx = label_keys.index(BEST_NEG_KEY) if BEST_NEG_KEY in label_keys else None

t0 = time.time()
for i in tqdm(range(0, len(texts), BATCH_SIZE), desc='감성 분석 중'):
    batch = texts[i:i+BATCH_SIZE]
    batch = [t if isinstance(t, str) and t.strip() else '내용 없음' for t in batch]
    inputs = tok(batch, return_tensors='pt', truncation=True,
                 padding=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        probs = torch.softmax(mod(**inputs).logits, dim=-1).cpu().numpy()
    for prob in probs:
        idx = int(np.argmax(prob))
        label_key = id2label[idx]
        sentiments.append(BEST_LABEL_MAP.get(label_key, label_key))
        scores.append(float(prob[idx]))
        score_pos_list.append(float(prob[pos_idx]) if pos_idx is not None else None)
        score_neg_list.append(float(prob[neg_idx]) if neg_idx is not None else None)

elapsed = time.time() - t0
print(f'\n✅ 감성분석 완료! 소요 시간: {elapsed/60:.1f}분')

df_result = df_full.copy()
df_result['sentiment'] = sentiments
df_result['sentiment_score'] = scores
df_result['score_positive'] = score_pos_list
df_result['score_negative'] = score_neg_list

print('\n감성 분포:')
print(df_result['sentiment'].value_counts())
df_result.head(3)

## [5단계] 결과 저장 및 다운로드
이 셀을 실행하면 `final_result.csv` 파일이 자동으로 다운로드됩니다.

In [ ]:
output_path = 'final_result.csv'
df_result.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f'✅ 저장 완료: {output_path} ({len(df_result):,}건)')

from google.colab import files
files.download(output_path)
print('다운로드가 시작됩니다. 브라우저 하단에서 파일을 확인하세요.')